# 2.10 Advanced Assignment Pitfalls

## Advanced / Expert-Level Python — Quantitative Finance and Data Science

**Notebook:** `2.10_Advanced_Assignment_Pitfalls.ipynb`  
**Python:** 3.10+ | **Dependencies:** Standard library only | **Scope:** Assignment-related correctness, state isolation, verification

### Overview

The most costly Python assignment bugs often produce **valid syntax, successful execution, and plausible numbers**. Their origin lies in the interaction of name binding with object identity, evaluation timing, mutable defaults, shared containers, scope rules, exception handling, and concurrency. These mechanisms can silently invalidate backtests, feature pipelines, risk reports, and experiment comparisons.

This notebook analyzes failure modes as **language semantics + counterexample + verified remedy**, not as stylistic warnings. Each executable example is followed by an **Analytical Interpretation** with implications for quantitative research. The capstone builds an **Isolation-Safe Point-in-Time Research Experiment Registry** with staged commits, deterministic results, immutable snapshots, and rollback tests.

### Learning Objectives

1. Distinguish a new binding from an independent object and a mutation.
2. Explain definition-time mutable defaults and dataclass factories.
3. Detect late-bound closure capture in parameter searches and callbacks.
4. Diagnose aliasing introduced by repetition, `dict.fromkeys`, and nested containers.
5. Evaluate shallow copying, deep copying, read-only views, and defensive snapshots.
6. Explain mutation-before-failure and partial assignment updates.
7. Identify non-atomic unpacking and iterator-consumption side effects.
8. Analyze assignment expressions, comprehension scopes, and stale names.
9. Distinguish missing values from valid zero and false values.
10. Explain class-state leakage, exception-target deletion, and shadowing.
11. Recognize shared-state races and unsafe read-modify-write sequences.
12. Construct a limited AST-based assignment-pitfall diagnostic.
13. Design reproducible, isolated point-in-time research experiments.
14. Verify financial invariants, batch rollback, and cross-experiment independence.

### Scope Boundaries

This chapter integrates mechanisms introduced in Sections 2.1–2.9. It focuses on **failure analysis and corrective engineering**; complete treatments of descriptors, closures, concurrency, and financial backtesting are developed in their dedicated chapters. Examples deliberately avoid NumPy and Pandas to expose core Python semantics.

### Verification Convention

`assert` is used to demonstrate executable claims, while `unittest` provides a repeatable test suite. Never treat assertions alone as an operational risk-control boundary: Python may disable them with optimization (`-O`). Numerical outcomes here are synthetic research demonstrations, not evidence of trading profitability.

## Notebook Roadmap

| Section | Focus |
|---|---|
| 2.10.1 | Mutable default arguments and definition-time evaluation |
| 2.10.2 | Dataclass defaults, factories, and class-variable leakage |
| 2.10.3 | Late-binding closures and parameter sweep corruption |
| 2.10.4 | Aliasing through container repetition and `dict.fromkeys` |
| 2.10.5 | Shallow copies, deep copies, and misleading isolation |
| 2.10.6 | Augmented assignment: mutation followed by failure |
| 2.10.7 | Partial unpacking and consumed iterators |
| 2.10.8 | Overlapping targets and left-to-right side effects |
| 2.10.9 | Walrus expressions, comprehension scope, and stale names |
| 2.10.10 | Truthiness, `None`, and sentinel-based assignment |
| 2.10.11 | Binding to mutable configuration and cache invalidation |
| 2.10.12 | Exception targets, shadowing, and pattern captures |
| 2.10.13 | Read-modify-write races and synchronization |
| 2.10.14 | Hashing, equality, and mutable identity traps |
| 2.10.15 | Static AST diagnostics: useful but limited |
| 2.10.16 | Capstone: Isolation-Safe Research Experiment Registry |
| 2.10.17 | Automated testing and numerical invariants |
| 2.10.18 | Final Analytical Interpretation, exercises, and challenge |

**Recommended cell order:** Execute from a freshly restarted kernel, top to bottom. Intentionally incorrect examples are caught or isolated so that the notebook completes without uncaught errors.

## 2.10.1 Mutable Default Arguments and Definition-Time Evaluation

### Theoretical Foundations

A function's ordinary default argument expression is evaluated **once, when the function definition executes**, not once per call. The resulting object is retained in the function's defaults. A mutable list or dictionary used as a default may therefore persist and accumulate changes across calls, even across nominally independent research runs.

**Wrong assumption:** `def collect(event, history=[]): ...` provides a fresh list for each call.

**Actual semantics:** the same list becomes the default on every invocation omitting `history`. A nonmutable default, such as `None`, plus explicit construction within the function, is a conventional remedy. A purposely stateful default can also be valid if its lifetime is documented and controlled.

**Additional pitfall:** a default that references a variable or calls a function captures the result at *definition time*, not its value on each call. Rebinding the original variable later does not update the default.

**Finance failure mode:** sequential simulations contaminate one another's collected trades, while changing a portfolio risk limit may have no effect on a function using a previously captured default.

In [1]:
# Intentionally flawed: one list is shared by omitted arguments.
def unsafe_collect(symbol: str, history: list[str] = []) -> list[str]:
    history.append(symbol)
    return history

first_run = unsafe_collect("AAPL")
second_run = unsafe_collect("MSFT")
assert first_run is second_run
assert second_run == ["AAPL", "MSFT"]

# Correct: allocate a fresh list when the argument is omitted.
def safe_collect(
    symbol: str,
    history: list[str] | None = None,
) -> list[str]:
    result = [] if history is None else history
    result.append(symbol)
    return result

isolated_a = safe_collect("AAPL")
isolated_b = safe_collect("MSFT")
assert isolated_a == ["AAPL"] and isolated_b == ["MSFT"]
assert isolated_a is not isolated_b

threshold = 0.25
def captured_default(value: float, limit: float = threshold) -> bool:
    return value <= limit

threshold = 0.10
assert captured_default(0.20) is True  # still uses 0.25
assert captured_default(0.20, limit=threshold) is False
print("Shared default:", second_run)
print("Independent runs:", isolated_a, isolated_b)
print("Definition-time limit:", captured_default.__defaults__)

Shared default: ['AAPL', 'MSFT']
Independent runs: ['AAPL'] ['MSFT']
Definition-time limit: (0.25,)


### **Analytical Interpretation**

**1. Definition-time evaluation** — The list is created when `unsafe_collect` is defined and retained in its default tuple.

**2. Aliasing across calls** — Omitting the optional argument reuses the same list; the first returned reference observes a later call’s mutation.

**3. Correct ownership** — A `None` sentinel separates omission from an explicitly supplied list. If a caller passes a list, the function still modifies it by design.

**4. Captured configuration** — The default threshold remains `0.25` after the outer name is rebound to `0.10`.

**5. Quant application** — Do not let independent strategy runs share implicit trade histories or stale policy parameters.

## 2.10.2 Dataclass Defaults, Factories, and Class Variables

### Theoretical Foundations

Dataclasses generate initialization machinery but **do not automatically make nested mutable objects independent**. For mutable per-instance data, use `field(default_factory=...)`. A plain class attribute is shared among instances unless an instance shadows it. In dataclasses, use `ClassVar` for intentionally shared class-level metadata.

Modern `dataclasses` reject many unhashable defaults (Python 3.11+ broadened the rule), but that check is **not** a complete deep-mutability detector. `frozen=True` restricts ordinary field reassignment, not mutation of objects reachable through a field.

**Finance failure mode:** constructing many scenario records that unexpectedly share one list of fills or an optimizer’s mutable coefficients.

In [2]:
from dataclasses import dataclass, field, FrozenInstanceError
from typing import ClassVar

@dataclass
class ResearchTrial:
    name: str
    observations: list[float] = field(default_factory=list)
    format_version: ClassVar[int] = 1

trial_a = ResearchTrial("baseline")
trial_b = ResearchTrial("stress")
trial_a.observations.append(0.02)
assert trial_a.observations == [0.02]
assert trial_b.observations == []
assert trial_a.observations is not trial_b.observations

class UnsafeTrial:
    events: list[str] = []  # shared class attribute

unsafe_a = UnsafeTrial()
unsafe_b = UnsafeTrial()
unsafe_a.events.append("fill-001")
assert unsafe_b.events == ["fill-001"]

@dataclass(frozen=True)
class FrozenButNestedMutable:
    labels: list[str] = field(default_factory=list)

frozen = FrozenButNestedMutable()
frozen.labels.append("revised")
assert frozen.labels == ["revised"]
try:
    frozen.labels = []
except FrozenInstanceError:
    pass
else:
    raise AssertionError("Frozen field assignment should fail")

print("Isolated dataclasses:", trial_a, trial_b)
print("Shared class state:", unsafe_b.events)
print("Frozen outer, mutable inner:", frozen)

Isolated dataclasses: ResearchTrial(name='baseline', observations=[0.02]) ResearchTrial(name='stress', observations=[])
Shared class state: ['fill-001']
Frozen outer, mutable inner: FrozenButNestedMutable(labels=['revised'])


### **Analytical Interpretation**

**1. Factory semantics** — `default_factory=list` executes for each instance and avoids cross-trial list aliasing.

**2. Class-level state** — The ordinary `UnsafeTrial.events` list is shared through attribute lookup by instances lacking their own `events` field.

**3. Frozen is shallow** — A frozen dataclass prevents ordinary field rebinding but does not make a nested list immutable.

**4. Version-sensitive guard** — Dataclass default validation changed in Python 3.11; do not rely on it to prove ownership or deep immutability.

**5. Quant application** — Store per-experiment observations in separately constructed containers; use immutable inner records where historical integrity matters.

## 2.10.3 Late-Binding Closures and Parameter-Sweep Corruption

### Theoretical Foundations

A closure generally resolves a free variable **when the function is called**, not when the closure is created. Functions constructed in a loop may therefore all refer to the loop variable’s final value. Merely changing the function's name or wrapping it in a `lambda` does not fix the capture.

Safe approaches include an enclosing factory function, `functools.partial`, or a default argument that intentionally captures an immutable value at creation. Be careful: capturing a mutable object by default preserves its reference, not its contents.

**Finance failure mode:** a grid search appears to compare multiple lookbacks but all generated signal functions evaluate the same final lookback, creating falsely independent experiment labels.

In [3]:
from functools import partial

lookbacks = (2, 3, 5)
broken_rules = [lambda x: x >= lookback for lookback in lookbacks]
observed_broken = [rule(4) for rule in broken_rules]
assert observed_broken == [False, False, False]

# Capture a new immutable value for every rule.
fixed_rules = [
    lambda x, lookback=lookback: x >= lookback
    for lookback in lookbacks
]
assert [rule(4) for rule in fixed_rules] == [True, True, False]

def threshold_rule(x: int, threshold: int) -> bool:
    return x >= threshold

partial_rules = [
    partial(threshold_rule, threshold=lookback)
    for lookback in lookbacks
]
assert [rule(4) for rule in partial_rules] == [True, True, False]

# Caution: a captured default can still reference mutable state.
mutable_parameter = {"cutoff": 3}
captured_reference = lambda x, cfg=mutable_parameter: x >= cfg["cutoff"]
mutable_parameter["cutoff"] = 8
assert captured_reference(4) is False

print("Late-bound rules:", observed_broken)
print("Early-bound values:", [r(4) for r in fixed_rules])

Late-bound rules: [False, False, False]
Early-bound values: [True, True, False]


### **Analytical Interpretation**

**1. Closure cell** — The broken callbacks refer to a single loop-variable binding and observe its last value `5`.

**2. Immutable capture** — A default argument captures each numeric lookback at function creation.

**3. Alternative binding** — `functools.partial` stores explicit argument bindings without closing over the loop name.

**4. Mutable capture trap** — Capturing a dictionary as a default captures its object reference; later dictionary mutation remains observable.

**5. Research validity** — Parameter sweeps require independent parameter values as well as independent output state, with tests for every configuration.

## 2.10.4 Aliasing Through Repetition and `dict.fromkeys`

### Theoretical Foundations

Sequence repetition (`[item] * n`) repeats **references** to `item`; it does not deep-copy a nested mutable object. Likewise `dict.fromkeys(keys, value)` assigns the **same value object** to every key. These constructs are safe when the repeated object is immutable or sharing is intentional, but dangerous for mutable rows, model buffers, and per-asset dictionaries.

Comprehensions that create a new object per iteration are a common alternative. Always inspect both outer and inner identity: distinct outer containers can still share nested data.

In [4]:
symbols = ("AAPL", "MSFT", "JPM")
broken_matrix = [[0, 0]] * 3
broken_matrix[0][0] = 9
assert [row[0] for row in broken_matrix] == [9, 9, 9]

independent_matrix = [[0, 0] for _ in range(3)]
independent_matrix[0][0] = 9
assert [row[0] for row in independent_matrix] == [9, 0, 0]

bad_history = dict.fromkeys(symbols, [])
bad_history["AAPL"].append("event-1")
assert all(history == ["event-1"] for history in bad_history.values())

safe_history = {symbol: [] for symbol in symbols}
safe_history["AAPL"].append("event-1")
assert safe_history["MSFT"] == []
assert len({id(v) for v in safe_history.values()}) == 3

print("Repeated references:", broken_matrix)
print("Independent matrix:", independent_matrix)
print("Per-ticker events:", safe_history)

Repeated references: [[9, 0], [9, 0], [9, 0]]
Independent matrix: [[9, 0], [0, 0], [0, 0]]
Per-ticker events: {'AAPL': ['event-1'], 'MSFT': [], 'JPM': []}


### **Analytical Interpretation**

**1. Reference multiplication** — `* 3` repeats three references to the same inner list.

**2. Independent construction** — The nested comprehension constructs a fresh inner list for each row.

**3. Mapping alias** — `dict.fromkeys` does not clone a mutable default value for each key.

**4. Safe per-asset state** — A dictionary comprehension allocates independent event lists.

**5. Finance consequence** — A single portfolio or feature revision can contaminate multiple assets when inner buffers are accidentally shared.

## 2.10.5 Shallow Copy, Deep Copy, and Misleading Isolation

### Theoretical Foundations

`copy.copy` reconstructs an outer container but normally retains references to nested objects. `copy.deepcopy` traverses supported object graphs and uses memoization to preserve alias relationships and cycles **inside the copied graph**. Neither operation constitutes a universal, cost-free, reliable snapshot of arbitrary resources (open files, locks, opaque extension objects, shared external services).

A `MappingProxyType` is a live read-only view over the dictionary it wraps. If another reference mutates the underlying dictionary, the proxy reflects that change. True research snapshots must independently capture the relevant data before exposing read-only access.

In [5]:
import copy
from types import MappingProxyType

shared_series = [100, 102]
graph = {"AAPL": shared_series, "REFERENCE": shared_series}
shallow = copy.copy(graph)
deep = copy.deepcopy(graph)

assert shallow is not graph
assert shallow["AAPL"] is graph["AAPL"]
assert deep["AAPL"] is not graph["AAPL"]
assert deep["AAPL"] is deep["REFERENCE"]  # alias preserved within copy

graph["AAPL"].append(104)
assert shallow["REFERENCE"] == [100, 102, 104]
assert deep["REFERENCE"] == [100, 102]

source = {"AAPL": 100}
live_proxy = MappingProxyType(source)
source["AAPL"] = 150
assert live_proxy["AAPL"] == 150

isolated_proxy = MappingProxyType(source.copy())
source["AAPL"] = 200
assert isolated_proxy["AAPL"] == 150

print("Shallow:", shallow)
print("Deep:", deep)
print("Live view:", live_proxy["AAPL"])
print("Copied view:", isolated_proxy["AAPL"])

Shallow: {'AAPL': [100, 102, 104], 'REFERENCE': [100, 102, 104]}
Deep: {'AAPL': [100, 102], 'REFERENCE': [100, 102]}
Live view: 200
Copied view: 150


### **Analytical Interpretation**

**1. Shallow aliasing** — Outer identity changes while inner references remain shared.

**2. Deep-copy graph** — `deepcopy` constructs separate inner objects but preserves relationships among multiple references to the same inner object.

**3. Live proxy** — A read-only proxy prevents writes through the proxy; it cannot freeze the wrapped dictionary.

**4. Independent capture** — Copying the dictionary before wrapping creates an isolated flat mapping in this simple example, but nested mutable values would still require treatment.

**5. Research design** — Choose explicit snapshots and provenance over blind copying of large market-data structures.

## 2.10.6 Augmented Assignment: Mutation Before Failure

### Theoretical Foundations

Augmented assignment is not universally equivalent to `x = x + y`. For a mutable target, `__iadd__` may mutate the original object. Python then performs the assignment back to the target, which can raise an exception **after the mutation**.

The canonical trap is augmenting a mutable list reachable from an immutable tuple. Catching `TypeError` does not undo earlier mutation. This extends to property setters, custom containers, and other target-assignment protocols.

In [6]:
record = ([100, 200],)
original_nested = record[0]
try:
    record[0] += [300]
except TypeError:
    pass
assert record[0] is original_nested
assert record[0] == [100, 200, 300]

class RejectingProperty:
    def __init__(self):
        self._items = [1]
    @property
    def items(self):
        return self._items
    @items.setter
    def items(self, value):
        raise ValueError("Reassignment prohibited")

target = RejectingProperty()
try:
    target.items += [2]
except ValueError:
    pass
assert target.items == [1, 2]

# A staged alternative avoids exposing the mutation before validation.
proposed = [*target.items, 3]
assert proposed == [1, 2, 3]
assert target.items == [1, 2]
print("Tuple-retained mutation:", record)
print("Property-retained mutation:", target.items)

Tuple-retained mutation: ([100, 200, 300],)
Property-retained mutation: [1, 2]


### **Analytical Interpretation**

**1. In-place mutation** — List `__iadd__` mutates the object before the subsequent assignment back to the tuple element.

**2. Assignment failure** — The immutable tuple target rejects assignment, but there is no automatic rollback of the mutated list.

**3. Property trap** — The property getter returns a mutable list which changes before the setter raises `ValueError`.

**4. Safer staging** — Construct a separate candidate list and validate before publishing it.

**5. Finance implication** — Failed ledger or risk-limit updates may still contaminate mutable nested state unless ownership and commit boundaries are explicit.

## 2.10.7 Partial Unpacking and Consumed Iterators

### Theoretical Foundations

Iterable unpacking consumes the right-hand iterable and assigns targets as required by its structure. A failed unpacking operation can **consume part or all of an iterator**, even when target bindings are not updated. The exact side effects depend on the iterator and target operations.

Do not assume that catching a `ValueError` restores a stream cursor, file position, or generated data. For replayable sources, materialize and validate a bounded batch; for large streams, use explicit transaction framing, acknowledgements, or retryable offsets.

In [7]:
def observed_generator(events: list[str]):
    for i in range(3):
        events.append(f"yield-{i}")
        yield i

consumed_events: list[str] = []
source_iter = observed_generator(consumed_events)
left, right = ("unchanged-L", "unchanged-R")
try:
    left, right = source_iter
except ValueError:
    pass
assert (left, right) == ("unchanged-L", "unchanged-R")
assert consumed_events == ["yield-0", "yield-1", "yield-2"]
assert list(source_iter) == []

# Extended unpacking handles unknown intermediate length.
first, *middle, last = iter((10, 20, 30, 40))
assert (first, middle, last) == (10, [20, 30], 40)
print("Consumed events:", consumed_events)
print("Bindings after failure:", left, right)

Consumed events: ['yield-0', 'yield-1', 'yield-2']
Bindings after failure: unchanged-L unchanged-R


### **Analytical Interpretation**

**1. Iterator side effects** — The attempt to unpack two values checks for extra elements, consuming the third value as well.

**2. Binding preservation** — For this length-mismatch case, the preexisting target bindings remain unchanged.

**3. Non-replayability** — The generator has been exhausted even though unpacking raised `ValueError`.

**4. Extended targets** — A starred target deliberately collects intermediate items into a list.

**5. Data engineering** — Never assume failed record parsing rewinds a live stream; use explicit buffering and offset tracking.

## 2.10.8 Overlapping Targets, Left-to-Right Writes, and Partial Failure

### Theoretical Foundations

In a standard assignment, the right-hand side is evaluated before target assignment, but **target assignments proceed left to right**. An earlier target may modify a name or object used by a later target. An exception on a later target can leave earlier mutations applied.

Unpacking is not a multi-location transaction. A critical application should stage and validate the entire proposed state rather than relying on simultaneous-looking assignment syntax.

In [8]:
values = [0, 1]
index = 0
index, values[index] = 1, 9
assert values == [0, 9]  # uses updated index

values2 = [0, 1]
index2 = 0
values2[index2], index2 = 9, 1
assert values2 == [9, 1]

class Protected:
    def __init__(self):
        self.cash = 100
        self._limit = 0.25
    @property
    def limit(self):
        return self._limit
    @limit.setter
    def limit(self, value):
        if not 0 <= value <= 1:
            raise ValueError("Bad limit")
        self._limit = value

state = Protected()
try:
    state.cash, state.limit = 80, 1.5
except ValueError:
    pass
assert state.cash == 80 and state.limit == 0.25
print("Overlapping targets:", values, values2)
print("Partial failure state:", state.cash, state.limit)

Overlapping targets: [0, 9] [9, 1]
Partial failure state: 80 0.25


### **Analytical Interpretation**

**1. Target order** — The first overlapping assignment rebinds `index` before evaluating the later subscription target.

**2. Reversed form** — Reordering targets changes which list element receives the value.

**3. Partial write** — The cash attribute is updated before the validated property rejects the new limit.

**4. No automatic rollback** — A caught exception does not reverse already-applied attribute or item assignments.

**5. Quant remedy** — Construct candidate ledger state separately and publish only after complete validation.

## 2.10.9 Assignment Expressions, Comprehension Scope, and Stale Names

### Theoretical Foundations

The assignment expression `:=` binds a name as part of evaluating an expression. Inside many comprehensions, a walrus target binds in the **containing scope**, not in the comprehension's iteration-variable scope. That name can remain bound after the comprehension, and may retain a value from a previous run when the new comprehension produces no binding.

A related error occurs with empty loops: a loop target is not newly bound if the iterable is empty. An old value of the same name may therefore remain visible. Never reuse a possibly stale name as evidence that a new calculation succeeded.

In [9]:
def walrus_leakage():
    last_value = "stale"
    results = [(last_value := x * 2) for x in (1, 2, 3)]
    return results, last_value

assert walrus_leakage() == ([2, 4, 6], 6)

def empty_walrus():
    last_value = "stale"
    results = [(last_value := x * 2) for x in ()]
    return results, last_value

assert empty_walrus() == ([], "stale")

def empty_loop():
    current = "old-run"
    for current in ():
        pass
    return current

assert empty_loop() == "old-run"

# Correct: explicit return value, no reliance on leaked target.
values = [x * 2 for x in ()]
last = values[-1] if values else None
assert last is None
print("Comprehension leakage:", walrus_leakage())
print("Empty binding:", empty_walrus())

Comprehension leakage: ([2, 4, 6], 6)
Empty binding: ([], 'stale')


### **Analytical Interpretation**

**1. Scope rule** — The walrus target in the demonstrated list comprehension writes to the containing function scope.

**2. Empty evaluation** — If the comprehension has no iterations, its assignment expression is never evaluated.

**3. Stale binding** — A prior binding can survive and be mistaken for a new result.

**4. Loop analogy** — An empty `for` loop similarly does not produce a new binding for its target.

**5. Safer API** — Use an explicit result object and represent missing observations with a deliberate sentinel such as `None`.

## 2.10.10 Truthiness, `None`, and Sentinel-Based Assignment

### Theoretical Foundations

The idioms `x = candidate or fallback`, `if x:`, and `dict.get(key) or fallback` conflate **missingness** with falsy values (`0`, `0.0`, `False`, `''`, empty containers). In financial software, zero cash flow, zero spread, and a zero realized return can be legitimate observations.

Use `is None` when the contract defines `None` as absent. For APIs where `None` itself is a valid stored value, construct a unique sentinel object and compare it by identity. Do not use `==` for sentinel identity.

In [10]:
EMPTY = object()
metrics = {"realized_pnl": 0.0, "optional": None}

incorrect = metrics.get("realized_pnl") or 100.0
assert incorrect == 100.0  # silently corrupts a valid zero

raw = metrics.get("realized_pnl")
correct = 100.0 if raw is None else raw
assert correct == 0.0

missing = metrics.get("not_present", EMPTY)
present_none = metrics.get("optional", EMPTY)
assert missing is EMPTY
assert present_none is None

# Explicit update preserves a legitimate zero.
def update_if_present(old: float, candidate: float | None) -> float:
    return old if candidate is None else candidate

assert update_if_present(5.0, 0.0) == 0.0
assert update_if_present(5.0, None) == 5.0
print("Bad zero fallback:", incorrect)
print("Valid zero preserved:", correct)

Bad zero fallback: 100.0
Valid zero preserved: 0.0


### **Analytical Interpretation**

**1. Truthiness bug** — `0.0 or 100.0` selects the fallback despite the zero being a valid P&L observation.

**2. Explicit absence** — Testing `is None` preserves values that are falsy but present.

**3. Unique sentinel** — An `object()` sentinel distinguishes a missing key from a key whose value is `None`.

**4. Identity semantics** — Use `is` for sentinels, not overloaded equality.

**5. Financial impact** — Incorrect fallback assignments can silently inflate risk metrics, transaction costs, or model results.

## 2.10.11 Mutating Configuration After Capturing a Binding

### Theoretical Foundations

A function can capture a reference to a mutable configuration, cache, or lookup table. Later changes to its contents are observable through that captured reference even if the enclosing name is rebound. Conversely, a shallowly frozen configuration can still reference mutable nested members.

Cache keys must include **every input and configuration dimension that affects a result**. Reusing a cached calculation across two differently configured experiments is a data-lineage error, not merely a performance concern.

In [11]:
from functools import lru_cache

mutable_config = {"multiplier": 2}
factor = lambda x, cfg=mutable_config: x * cfg["multiplier"]
assert factor(3) == 6
mutable_config["multiplier"] = 5
assert factor(3) == 15

@lru_cache(maxsize=32)
def invalid_cache(x: int) -> int:
    return x * mutable_config["multiplier"]

assert invalid_cache(3) == 15
mutable_config["multiplier"] = 7
assert invalid_cache(3) == 15  # stale cached result

# All economically material parameters belong in the key.
@lru_cache(maxsize=32)
def valid_cache(x: int, multiplier: int) -> int:
    return x * multiplier

assert valid_cache(3, 7) == 21
assert valid_cache(3, 5) == 15
print("Stale cached value:", invalid_cache(3))
print("Parameter-complete cache:", valid_cache(3, 7))

Stale cached value: 15
Parameter-complete cache: 21


### **Analytical Interpretation**

**1. Captured reference** — A function default stores the configuration dictionary object, not a frozen copy of its contents.

**2. Cache contamination** — The cached function’s key excludes a configuration variable used in its computation; it returns a stale value after mutation.

**3. Explicit dependency** — The corrected cache includes the multiplier as a hashable argument, providing parameter-specific results.

**4. Research provenance** — Market data versions, lookback lengths, transaction costs, and signal thresholds must enter reproducible research keys.

## 2.10.12 Exception Targets, Built-in Shadowing, and Pattern Captures

### Theoretical Foundations

Python clears an exception handler's `except ... as name` binding at the end of the handler, to help avoid traceback reference cycles. If the same name previously existed in the function, the handler can remove that binding rather than restore its former value.

`match` capture patterns also **bind names**, including when an associated guard evaluates false. Python does not define all partial-binding details for unsuccessful pattern matching, so do not use bindings from a failed match as valid data. The safer approach is to read the candidate record from an explicit return value after successful validation.

Separately, assigning to `sum`, `list`, `id`, or `type` shadows useful built-in names in that scope.

In [12]:
def exception_binding_deletion():
    error = "previous-value"
    try:
        raise ValueError("bad market record")
    except ValueError as error:
        assert isinstance(error, ValueError)
    try:
        return error
    except UnboundLocalError:
        return "handler-binding-cleared"

assert exception_binding_deletion() == "handler-binding-cleared"

def guarded_match(record: object) -> tuple[str, float] | None:
    match record:
        case {"ticker": str(ticker), "price": (int() | float()) as price} if price > 0:
            return ticker, float(price)
        case _:
            return None

assert guarded_match({"ticker": "AAPL", "price": 185.0}) == ("AAPL", 185.0)
assert guarded_match({"ticker": "AAPL", "price": -1.0}) is None

def shadow_builtin():
    sum = 100  # local name hides the built-in function
    try:
        sum((1, 2))
    except TypeError:
        return "shadowed"
    return "not-shadowed"

assert shadow_builtin() == "shadowed"
print("Exception binding:", exception_binding_deletion())
print("Guarded matching:", guarded_match({"ticker": "AAPL", "price": 185.0}))

Exception binding: handler-binding-cleared
Guarded matching: ('AAPL', 185.0)


### **Analytical Interpretation**

**1. Handler cleanup** — The exception binding is cleared after the handler completes; its former local value is not restored.

**2. Guard discipline** — The sample only returns a validated match after its guard succeeds; it never inspects capture names following failure.

**3. Pattern limitations** — Unsuccessful matches have implementation-sensitive partial-binding details; relying on such names is unsafe.

**4. Built-in shadowing** — A local integer named `sum` prevents ordinary access to the built-in function through that name.

**5. Engineering principle** — Maintain explicit validated results, avoid ambiguous namespace reuse, and never use except-handler names to preserve unrelated state.

## 2.10.13 Shared-State Read–Modify–Write Races

### Theoretical Foundations

`counter += 1` is a read–modify–write operation at the language level. Its safety depends on the object, runtime, and synchronization policy. A compound sequence of reads and assignments is not an application-level transaction merely because it occupies one line.

To demonstrate a lost update **deterministically** without relying on an interpreter-specific scheduler, synchronize two workers so that both read the same old value before either writes its update. Then compare to an explicitly locked version.

In [13]:
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier, Lock

def lost_update_demo() -> int:
    shared = {"count": 0}
    rendezvous = Barrier(2)
    def worker() -> None:
        observed = shared["count"]
        rendezvous.wait(timeout=5)
        shared["count"] = observed + 1
    with ThreadPoolExecutor(max_workers=2) as pool:
        list(pool.map(lambda _: worker(), range(2)))
    return shared["count"]

assert lost_update_demo() == 1

def synchronized_demo() -> int:
    shared = {"count": 0}
    guard = Lock()
    def worker() -> None:
        with guard:
            shared["count"] = shared["count"] + 1
    with ThreadPoolExecutor(max_workers=2) as pool:
        list(pool.map(lambda _: worker(), range(2)))
    return shared["count"]

assert synchronized_demo() == 2
print("Deliberately lost update:", lost_update_demo())
print("Locked update:", synchronized_demo())

Deliberately lost update: 1
Locked update: 2


### **Analytical Interpretation**

**1. Deterministic collision** — The barrier ensures both workers read zero before either writes, producing one retained increment.

**2. Synchronization** — The lock protects the complete read–modify–write critical section in the corrected example.

**3. Interpreter independence** — The example does not assume a specific GIL behavior or thread scheduling order.

**4. Quant relevance** — Concurrent cash, exposure, or fill counters require an explicit consistency policy, not merely assignment syntax.

## 2.10.14 Hashing, Equality, and Mutable Identity Traps

### Theoretical Foundations

A hashable object must obey the equality–hash contract: objects considered equal should have equal hashes, and an object used as a mapping key should not change its hash-relevant state while stored. Mutable objects are typically inappropriate as dictionary keys unless equality and hashing are explicitly defined to preserve invariants.

A more subtle issue is `NaN`: IEEE floating-point NaN compares unequal to itself (`nan != nan`), so it is inappropriate as a missing-value sentinel when ordinary equality semantics are expected. Signed zeros compare equal even though certain numerical operations can distinguish their signs.

In [14]:
from dataclasses import dataclass
from math import copysign, isnan

@dataclass(frozen=True)
class AssetKey:
    ticker: str
    venue: str

key_a = AssetKey("AAPL", "NASDAQ")
key_b = AssetKey("AAPL", "NASDAQ")
assert key_a == key_b and hash(key_a) == hash(key_b)
result_cache = {key_a: 100}
assert result_cache[key_b] == 100

nan = float("nan")
assert isnan(nan)
assert nan != nan

assert 0.0 == -0.0
assert copysign(1.0, 0.0) != copysign(1.0, -0.0)

missing = object()
assert missing is missing
print("Hashable immutable key:", key_a)
print("NaN self-equality:", nan == nan)
print("Signed zero equals:", 0.0 == -0.0)

Hashable immutable key: AssetKey(ticker='AAPL', venue='NASDAQ')
NaN self-equality: False
Signed zero equals: True


### **Analytical Interpretation**

**1. Immutable key** — A frozen dataclass with immutable fields supports stable value-based hashing for dictionary lookups.

**2. NaN semantics** — NaN is not equal to itself and must not be treated as an ordinary equality-based missing sentinel.

**3. Signed zero** — Positive and negative zero are equal, yet their signs can remain observable in numerical operations.

**4. Finance effect** — Cache keys and missing-price handling must rely on documented numerical semantics and immutable, normalized identities.

## 2.10.15 Static AST Diagnostics: Useful but Limited

### Theoretical Foundations

AST analysis can flag **syntactic patterns associated with assignment bugs**, including mutable function defaults, repeated nested-list literals, and `dict.fromkeys` calls with mutable literal values. It cannot prove that a function mutates a default, identify all possible aliases, or establish that a program is free from state contamination.

The demonstration below analyzes source text without executing it. It reports only selected patterns and associated source lines. Findings should be treated as review prompts, not definitive security or correctness verdicts.

In [15]:
import ast

class AssignmentPitfallInspector(ast.NodeVisitor):
    """Report selected syntactic assignment-risk patterns."""

    def __init__(self):
        self.findings: list[tuple[int, str]] = []

    def visit_FunctionDef(self, node: ast.FunctionDef):
        for default in (*node.args.defaults, *(
            d for d in node.args.kw_defaults if d is not None
        )):
            if isinstance(default, (ast.List, ast.Dict, ast.Set)):
                self.findings.append((node.lineno, "mutable-default-literal"))
        self.generic_visit(node)

    def visit_BinOp(self, node: ast.BinOp):
        if isinstance(node.op, ast.Mult):
            if isinstance(node.left, ast.List) and any(
                isinstance(x, (ast.List, ast.Dict, ast.Set))
                for x in node.left.elts
            ):
                self.findings.append((node.lineno, "nested-list-repetition"))
        self.generic_visit(node)

    def visit_Call(self, node: ast.Call):
        if (
            isinstance(node.func, ast.Attribute)
            and node.func.attr == "fromkeys"
            and len(node.args) >= 2
            and isinstance(node.args[1], (ast.List, ast.Dict, ast.Set))
        ):
            self.findings.append((node.lineno, "fromkeys-mutable-value"))
        self.generic_visit(node)

suspicious_source = """
def collect(item, output=[]):
    output.append(item)
    return output
x = [[0]] * 3
y = dict.fromkeys(('AAPL', 'MSFT'), [])
"""

inspector = AssignmentPitfallInspector()
inspector.visit(ast.parse(suspicious_source))
assert {kind for _, kind in inspector.findings} == {
    "mutable-default-literal", "nested-list-repetition", "fromkeys-mutable-value"
}
print("Static findings:", inspector.findings)

Static findings: [(2, 'mutable-default-literal'), (5, 'nested-list-repetition'), (6, 'fromkeys-mutable-value')]


### **Analytical Interpretation**

**1. AST-level signal** — The visitor recognizes three concrete source patterns without running the program.

**2. False positives** — A mutable default can be intentionally shared and never mutated; syntax alone is not proof of a bug.

**3. False negatives** — Aliases constructed through factories, indirect functions, or runtime reflection may not match these narrow patterns.

**4. Tooling value** — Static warnings should be combined with runtime invariants, tests, and data ownership reviews.

## 2.10.16 Practical Application — Isolation-Safe Point-in-Time Research Experiment Registry

### Research Problem

A quantitative researcher runs multiple momentum experiments with different lookback windows and thresholds on a mutable price dataset. Results are stored in a shared registry. The research process is vulnerable to **five distinct assignment-related defects**:

1. **Source aliasing:** a later price correction rewrites past experiment inputs.
2. **Late binding:** all callbacks silently reference the final lookback in a loop.
3. **Mutable shared results:** results from one experiment are modified by another.
4. **Incomplete cache/experiment keys:** a distinct parameter set is mislabeled as a previously computed experiment.
5. **Partial batch commit:** one invalid experiment leaves earlier results visible in a supposedly atomic batch.

### Architecture

    Mutable raw prices
          |
          v
    Strict Decimal validation
          |
          v
    Immutable point-in-time snapshot + SHA-256 fingerprint
          |
          v
    Immutable experiment specifications
          |
          v
    Signal formation at session t (past and present only)
          |
          v
    Evaluation at t+1 entry and t+2 exit
          |
          v
    Staged immutable experiment results
          |
      +---+---+
      |       |
    Failure  Success
      |       |
    Discard Commit under lock

### Model Definitions

For ticker `i` and lookback `L`:

\[
M_{i,t} = \frac{P_{i,t}}{P_{i,t-L}} - 1
\]

The signal is LONG if momentum is at least `threshold`, SHORT if it is at most negative `threshold`, and otherwise FLAT. The signal is computed from information through session `t`; hypothetical entry and exit closes are `t+1` and `t+2`. For a hypothetical **fixed evaluation notional** `N = $10,000`:

\[
\mathrm{GrossPnL} = N \cdot d \left(\frac{P_{t+2}}{P_{t+1}}-1\right)
\]

\[
\mathrm{NetPnL} = \mathrm{GrossPnL} - N \cdot \frac{\mathrm{RoundTripCostBps}}{10{,}000}
\]

These are **signal-level hypothetical P&Ls**, **not a portfolio NAV or compounded backtest**. Overlapping evaluations may reuse capital; borrowing, short availability, slippage realization, corporate actions, and actual execution feasibility are not modeled.

### Integrity Contracts

- Inputs use `Decimal` prices, never `float` silently converted to `Decimal`.
- Every ticker has the same consecutive session index in this synthetic example.
- Snapshot records, trade evaluations, and experiment outcomes are immutable.
- Source revisions cannot affect existing snapshots.
- Every experiment has an explicit unique identifier and frozen parameter contract.
- A batch publishes either all its results or none under the registry lock.
- Rejected batches do not advance registry version.
- Experiment outcomes cannot depend on other experiments' execution order.
- The same input snapshot and spec produce the same result.

In [16]:
# ============================================================
# CAPSTONE — VALIDATED IMMUTABLE RESEARCH DATA
# ============================================================
import hashlib
import json
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from decimal import Decimal, localcontext
from threading import RLock
from typing import Literal

D = Decimal
EVAL_NOTIONAL = D("10000")
BASIS_POINTS = D("10000")

@dataclass(frozen=True, slots=True)
class PriceObservation:
    ticker: str
    session: int
    close_usd: Decimal

    def __post_init__(self):
        if not isinstance(self.ticker, str) or not self.ticker.isalpha() or self.ticker != self.ticker.upper():
            raise ValueError("Ticker must contain uppercase ASCII letters")
        if not self.ticker.isascii():
            raise ValueError("Ticker must be ASCII")
        if type(self.session) is not int or self.session < 0:
            raise ValueError("Session must be a nonnegative integer")
        if type(self.close_usd) is not Decimal or not self.close_usd.is_finite() or self.close_usd <= 0:
            raise ValueError("Close must be a positive finite Decimal")

@dataclass(frozen=True, slots=True)
class MarketSnapshot:
    observations: tuple[PriceObservation, ...]
    fingerprint: str
    session_count: int

    @classmethod
    def from_prices(cls, raw: Mapping[str, Sequence[Decimal]]) -> 'MarketSnapshot':
        if not isinstance(raw, Mapping) or not raw:
            raise ValueError("Prices must be a nonempty mapping")
        lengths = {len(seq) for seq in raw.values()}
        if len(lengths) != 1 or min(lengths) < 5:
            raise ValueError("All assets must have equal histories of at least five sessions")
        records = []
        for ticker, prices in sorted(raw.items()):
            if isinstance(prices, (str, bytes)) or not isinstance(prices, Sequence):
                raise TypeError("Price history must be a sequence")
            for session, close in enumerate(prices):
                records.append(PriceObservation(ticker, session, close))
        frozen = tuple(records)
        serialized = json.dumps(
            [(r.ticker, r.session, str(r.close_usd)) for r in frozen],
            ensure_ascii=True,
            separators=(",", ":"),
        )
        fingerprint = hashlib.sha256(serialized.encode('ascii')).hexdigest()
        return cls(frozen, fingerprint, min(lengths))

    def price_series(self, ticker: str) -> tuple[Decimal, ...]:
        prices = tuple(r.close_usd for r in self.observations if r.ticker == ticker)
        if not prices:
            raise KeyError(ticker)
        return prices

@dataclass(frozen=True, slots=True)
class ExperimentSpec:
    experiment_id: str
    ticker: str
    lookback: int
    threshold: Decimal
    round_trip_cost_bps: Decimal

    def __post_init__(self):
        if not isinstance(self.experiment_id, str) or not self.experiment_id.strip():
            raise ValueError("Experiment ID must be nonempty")
        if not isinstance(self.ticker, str) or not self.ticker.isascii() or not self.ticker.isalpha() or self.ticker != self.ticker.upper():
            raise ValueError("Invalid ticker")
        if type(self.lookback) is not int or self.lookback < 1:
            raise ValueError("Lookback must be a positive integer")
        if type(self.threshold) is not Decimal or not self.threshold.is_finite() or not D(0) < self.threshold < D(1):
            raise ValueError("Threshold must be a finite Decimal in (0,1)")
        if type(self.round_trip_cost_bps) is not Decimal or not self.round_trip_cost_bps.is_finite() or self.round_trip_cost_bps < 0:
            raise ValueError("Cost must be a finite nonnegative Decimal")

@dataclass(frozen=True, slots=True)
class SignalEvaluation:
    decision_session: int
    entry_session: int
    exit_session: int
    direction: Literal['LONG', 'SHORT']
    momentum: Decimal
    gross_pnl_usd: Decimal
    transaction_cost_usd: Decimal
    net_pnl_usd: Decimal

@dataclass(frozen=True, slots=True)
class ExperimentResult:
    spec: ExperimentSpec
    data_fingerprint: str
    evaluations: tuple[SignalEvaluation, ...]
    gross_pnl_usd: Decimal
    total_cost_usd: Decimal
    net_pnl_usd: Decimal
    mean_net_return_bps: Decimal | None

@dataclass(frozen=True, slots=True)
class RegistrySnapshot:
    version: int
    results: tuple[tuple[str, ExperimentResult], ...]

print("Immutable market and research models ready")

Immutable market and research models ready


### **Analytical Interpretation**

**1. Independent data capture** — `MarketSnapshot.from_prices` builds immutable records from validated `Decimal` inputs rather than storing the raw price lists.

**2. Content identity** — A SHA-256 fingerprint identifies the canonical serialized observation sequence used by the experiment.

**3. Typed parameters** — The frozen spec explicitly records ticker, lookback, threshold, and round-trip cost; no hidden defaults or shared mutable configuration.

**4. Immutable output** — Every trade evaluation and experiment result is a frozen record, eliminating nested mutable output buffers.

**5. Scope and limitation** — The session-index convention is synthetic and does not encode calendar gaps, publication delays, or actual fill mechanics.

In [17]:
# ============================================================
# CAPSTONE — PURE RESEARCH EVALUATION
# ============================================================
def evaluate_experiment(
    market: MarketSnapshot,
    spec: ExperimentSpec,
) -> ExperimentResult:
    """Evaluate historical momentum with delayed hypothetical execution."""
    prices = market.price_series(spec.ticker)
    if len(prices) < spec.lookback + 3:
        raise ValueError("Insufficient past and future observations")

    evaluations: list[SignalEvaluation] = []
    # Local Decimal precision is explicit and isolated.
    with localcontext() as ctx:
        ctx.prec = 38
        per_trade_cost = EVAL_NOTIONAL * spec.round_trip_cost_bps / BASIS_POINTS
        for t in range(spec.lookback, len(prices) - 2):
            momentum = prices[t] / prices[t - spec.lookback] - D(1)
            if momentum >= spec.threshold:
                direction: Literal['LONG', 'SHORT'] = 'LONG'
                sign = D(1)
            elif momentum <= -spec.threshold:
                direction = 'SHORT'
                sign = D(-1)
            else:
                continue
            gross = EVAL_NOTIONAL * sign * (prices[t + 2] / prices[t + 1] - D(1))
            net = gross - per_trade_cost
            if not all(x.is_finite() for x in (momentum, gross, per_trade_cost, net)):
                raise OverflowError('Nonfinite research result')
            evaluations.append(SignalEvaluation(
                decision_session=t,
                entry_session=t + 1,
                exit_session=t + 2,
                direction=direction,
                momentum=+momentum,
                gross_pnl_usd=+gross,
                transaction_cost_usd=+per_trade_cost,
                net_pnl_usd=+net,
            ))
        gross_total = sum((r.gross_pnl_usd for r in evaluations), D(0))
        cost_total = sum((r.transaction_cost_usd for r in evaluations), D(0))
        net_total = sum((r.net_pnl_usd for r in evaluations), D(0))
        mean_bps = (
            net_total / (EVAL_NOTIONAL * len(evaluations)) * BASIS_POINTS
            if evaluations else None
        )
        return ExperimentResult(
            spec=spec,
            data_fingerprint=market.fingerprint,
            evaluations=tuple(evaluations),
            gross_pnl_usd=+gross_total,
            total_cost_usd=+cost_total,
            net_pnl_usd=+net_total,
            mean_net_return_bps=+mean_bps if mean_bps is not None else None,
        )

class ResearchRegistry:
    """In-process staged batch registry with immutable snapshots."""
    def __init__(self):
        self._results: dict[str, ExperimentResult] = {}
        self._version = 0
        self._lock = RLock()

    def snapshot(self) -> RegistrySnapshot:
        with self._lock:
            return RegistrySnapshot(
                version=self._version,
                results=tuple(sorted(self._results.items())),
            )

    def submit_batch(
        self, market: MarketSnapshot, specs: Sequence[ExperimentSpec]
    ) -> RegistrySnapshot:
        if not specs:
            raise ValueError('Batch cannot be empty')
        with self._lock:
            staged = self._results.copy()
            for spec in specs:
                if not isinstance(spec, ExperimentSpec):
                    raise TypeError('Every specification must be ExperimentSpec')
                if spec.experiment_id in staged:
                    raise ValueError(f'Duplicate experiment: {spec.experiment_id}')
                staged[spec.experiment_id] = evaluate_experiment(market, spec)
            # The staged dict is private until the complete batch succeeds.
            self._results = staged
            self._version += 1
            return self.snapshot()

print("Pure evaluation and staged experiment registry ready")

Pure evaluation and staged experiment registry ready


### **Analytical Interpretation**

**1. Pure evaluation** — `evaluate_experiment` depends only on an immutable market snapshot and immutable spec, not hidden mutable configuration or callback state.

**2. Point-in-time separation** — Momentum reads prices through session `t`; later prices are used only for retrospective entry and outcome evaluation.

**3. Staged submission** — Batch work modifies an independent dictionary, then publishes it after all specifications succeed.

**4. Duplicate protection** — Experiment identifiers cannot be reused within a batch or against already committed registry records.

**5. Limits** — The registry protects cooperating in-process reads and writes, but is not crash durable or a distributed database transaction.

In [18]:
# ============================================================
# CAPSTONE — DETERMINISTIC EXPERIMENT RESULTS
# ============================================================
def make_synthetic_prices():
    return {
        'AAPL': [D(v) for v in (100, 103, 106, 104, 108, 110, 107, 111, 113)],
        'MSFT': [D(v) for v in (200, 198, 196, 194, 197, 195, 193, 196, 198)],
    }

raw_market = make_synthetic_prices()
market_v1 = MarketSnapshot.from_prices(raw_market)
specs = (
    ExperimentSpec('AAPL_L2', 'AAPL', 2, D('0.015'), D('5')),
    ExperimentSpec('AAPL_L3', 'AAPL', 3, D('0.025'), D('8')),
    ExperimentSpec('MSFT_L2', 'MSFT', 2, D('0.010'), D('6')),
    ExperimentSpec('AAPL_L1', 'AAPL', 1, D('0.020'), D('4')),
)

registry = ResearchRegistry()
before_commit = registry.snapshot()
after_commit = registry.submit_batch(market_v1, specs)
by_id = dict(after_commit.results)
assert before_commit.version == 0 and after_commit.version == 1
assert len(by_id) == len(specs) == 4
assert {key: res.spec.lookback for key, res in by_id.items()} == {
    'AAPL_L2': 2, 'AAPL_L3': 3, 'MSFT_L2': 2, 'AAPL_L1': 1
}

print('ISOLATED POINT-IN-TIME RESEARCH REPORT')
print('-' * 88)
print(f"{'Experiment':<14}{'Trades':>7}{'Gross P&L':>18}{'Costs':>12}{'Net P&L':>18}")
for experiment_id, result in after_commit.results:
    print(f'{experiment_id:<14}{len(result.evaluations):>7}'
          f'{result.gross_pnl_usd:>18,.4f}'
          f'{result.total_cost_usd:>12,.2f}'
          f'{result.net_pnl_usd:>18,.4f}')
print('Committed Registry Version:', after_commit.version)
print('Dataset Fingerprint:', market_v1.fingerprint[:16] + '...')

ISOLATED POINT-IN-TIME RESEARCH REPORT
----------------------------------------------------------------------------------------
Experiment     Trades         Gross P&L       Costs           Net P&L
AAPL_L1             4         -256.9713       16.00         -272.9713
AAPL_L2             3          485.7199       15.00          470.7199
AAPL_L3             4          466.4699       32.00          434.4699
MSFT_L2             3         -155.1571       18.00         -173.1571
Committed Registry Version: 1
Dataset Fingerprint: ce24a18add7d8652...


### **Analytical Interpretation**

**1. Parameter independence** — Each experiment reports its own ticker, lookback, and threshold without accidentally capturing the final loop variable.

**2. Stable financial state** — All results store the same input fingerprint but distinct immutable parameter contracts.

**3. Trade-level accounting** — Every evaluation deducts its own proportional round-trip cost from gross P&L.

**4. Economics vs. methodology** — Positive or negative outcomes are artifacts of synthetic prices and simplified assumptions, not statistical evidence.

In [19]:
# ============================================================
# CAPSTONE — SOURCE MUTATION AND FAILED BATCH PROTECTION
# ============================================================
original_result = by_id['AAPL_L2']
immutable_before = registry.snapshot()
original_data_fingerprint = market_v1.fingerprint

# Change the mutable input *after* constructing the snapshot.
raw_market['AAPL'][-1] = D('500')
assert market_v1.price_series('AAPL')[-1] == D('113')
assert market_v1.fingerprint == original_data_fingerprint
assert evaluate_experiment(market_v1, specs[0]) == original_result

market_v2 = MarketSnapshot.from_prices(raw_market)
assert market_v2.fingerprint != market_v1.fingerprint

# In this batch, the first experiment would succeed; the second fails.
invalid_batch = (
    ExperimentSpec('VALID_UNCOMMITTED', 'AAPL', 2, D('0.02'), D('5')),
    ExperimentSpec('INVALID_HISTORY', 'MSFT', 7, D('0.02'), D('5')),
)
try:
    registry.submit_batch(market_v1, invalid_batch)
except ValueError as error:
    failure_message = str(error)
else:
    raise AssertionError('Invalid batch must be rejected')

assert registry.snapshot() == immutable_before
assert 'VALID_UNCOMMITTED' not in dict(registry.snapshot().results)
assert market_v2.price_series('AAPL')[-1] == D('500')
print('Original snapshot AAPL last:', market_v1.price_series('AAPL')[-1])
print('Revised snapshot AAPL last:', market_v2.price_series('AAPL')[-1])
print('Rejected batch reason:', failure_message)
print('Failed batch left committed results unchanged:', registry.snapshot() == immutable_before)

Original snapshot AAPL last: 113
Revised snapshot AAPL last: 500
Rejected batch reason: Insufficient past and future observations
Failed batch left committed results unchanged: True


### **Analytical Interpretation**

**1. Data isolation** — Changing raw market data cannot mutate previously captured immutable observations.

**2. Version fingerprints** — A source revision produces a distinct snapshot fingerprint.

**3. Partial-batch counterexample** — The first spec succeeds in staging; the second lacks enough observations and raises an exception.

**4. Rollback invariant** — Because only staged data was modified, the committed registry and version remain identical to their prior snapshot.

**5. Quant implication** — The framework prevents several common assignment-related contamination paths but does not prove backtest validity or durable storage consistency.

### Expected Results and Independent Financial Reconciliation

The synthetic experiments are not intended to optimize a strategy. They are designed to make **isolation and accounting invariants** observable.

| Experiment | Lookback | Trade Evaluations | Cost per Evaluation | Expected Aggregate Gross P&L (approx.) | Expected Net P&L (approx.) |
|---|---:|---:|---:|---:|---:|
| `AAPL_L2` | 2 | 3 | $5 | +$485.7199 | +$470.7199 |
| `AAPL_L3` | 3 | 4 | $8 | +$466.4699 | +$434.4699 |
| `MSFT_L2` | 2 | 3 | $6 | −$155.1571 | −$173.1571 |
| `AAPL_L1` | 1 | 4 | $4 | −$256.9713 | −$272.9713 |

These values are presented at four decimal places for readability; the engine retains higher-precision `Decimal` values.

#### Required Invariants

1. `net_pnl_usd == gross_pnl_usd - total_cost_usd` for every experiment, within its arithmetic context.
2. Every evaluated trade has `decision_session < entry_session < exit_session`.
3. A rewritten raw price must not change an existing `MarketSnapshot`.
4. Identical snapshot and specification must produce identical results.
5. A failed batch must leave registry version and committed records unchanged.
6. Duplicate experiment identifiers must never overwrite earlier results.
7. A no-trade result must use `mean_net_return_bps = None`, not manufacture a realized zero return.
8. Data fingerprints identify encoded input content; they do not prove provenance, market accuracy, or execution feasibility.

**Important:** Aggregated hypothetical trade P&L is not an equity curve, total portfolio return, or annualized performance figure. Some evaluated periods overlap.

## 2.10.17 Automated Testing and Numerical Invariants

### Testing Methodology

The suite has two layers: **language regression tests** for Python-specific pitfalls and **capstone invariants** for data integrity and financial results. Tests use freshly constructed inputs to avoid relying on notebook execution history.

| Category | Critical invariant |
|---|---|
| Mutable defaults | Omitted histories must not leak across calls |
| Closure binding | Every generated rule retains intended parameters |
| Nested aliasing | Independent rows and per-asset buffers must remain isolated |
| Partial failure | Failed assignment must not be mistaken for rollback |
| Missingness | Valid zero must not be replaced by fallback |
| Static inspection | Selected risky source patterns must be detected |
| Research input | Invalid price/type/history must be rejected |
| Parameter contract | Invalid ID, ticker, lookback, threshold, cost must be rejected |
| Temporal structure | Decision, entry, exit sessions strictly ordered |
| Financial accounting | Gross − cost = net; deterministic reference numbers |
| Registry | All-or-nothing staging, duplicate rejection, version discipline |
| Snapshot | Historical results persist after source changes |

The tests explicitly cover zero-trade behavior, multiple experiments with different parameters, and a failed batch whose first staged result was valid.

In [20]:
import unittest
from dataclasses import FrozenInstanceError

class AssignmentPitfallRegressionTests(unittest.TestCase):
    def test_mutable_default_is_shared(self):
        def bad(x, xs=[]):
            xs.append(x)
            return xs
        self.assertIs(bad(1), bad(2))
    def test_safe_default_isolated(self):
        self.assertIsNot(safe_collect('X'), safe_collect('Y'))
    def test_definition_time_default(self):
        self.assertTrue(captured_default(0.20))
    def test_dataclass_factory(self):
        a, b = ResearchTrial('a'), ResearchTrial('b')
        self.assertIsNot(a.observations, b.observations)
    def test_class_attribute_sharing(self):
        self.assertIs(UnsafeTrial().events, UnsafeTrial().events)
    def test_frozen_nested_mutability(self):
        obj = FrozenButNestedMutable()
        obj.labels.append('x')
        self.assertEqual(obj.labels, ['x'])
    def test_late_bound_loop(self):
        self.assertEqual([f(4) for f in broken_rules], [False]*3)
    def test_fixed_capture(self):
        self.assertEqual([f(4) for f in fixed_rules], [True, True, False])
    def test_partial_capture(self):
        self.assertEqual([f(4) for f in partial_rules], [True, True, False])
    def test_list_multiplication_alias(self):
        x = [[0]]*3
        self.assertIs(x[0],x[2])
    def test_list_comprehension_isolated(self):
        x = [[0] for _ in range(3)]
        self.assertIsNot(x[0],x[2])
    def test_fromkeys_alias(self):
        x = dict.fromkeys(('A','B'),[])
        self.assertIs(x['A'],x['B'])
    def test_copy_nested_alias(self):
        original = {'x':[1]}
        self.assertIs(copy.copy(original)['x'],original['x'])
    def test_deepcopy_independence(self):
        original = {'x':[1]}
        self.assertIsNot(copy.deepcopy(original)['x'],original['x'])
    def test_deepcopy_preserves_internal_alias(self):
        item = []
        clone = copy.deepcopy([item,item])
        self.assertIs(clone[0],clone[1])
    def test_mapping_proxy_live_view(self):
        src={'a':1}; p=MappingProxyType(src); src['a']=2
        self.assertEqual(p['a'],2)
    def test_augmented_tuple_mutates(self):
        value=([1],)
        with self.assertRaises(TypeError):
            value[0] += [2]
        self.assertEqual(value[0],[1,2])
    def test_augmented_property_mutates(self):
        obj=RejectingProperty()
        with self.assertRaises(ValueError):
            obj.items += [2]
        self.assertEqual(obj.items,[1,2])
    def test_failed_unpack_consumes(self):
        seq=iter((1,2,3))
        with self.assertRaises(ValueError):
            a,b=seq
        self.assertEqual(list(seq),[])
    def test_overlapping_targets(self):
        arr=[0,1]; idx=0
        idx,arr[idx]=1,7
        self.assertEqual(arr,[0,7])
    def test_partial_attribute_update(self):
        obj=Protected()
        with self.assertRaises(ValueError):
            obj.cash,obj.limit=80,1.5
        self.assertEqual((obj.cash,obj.limit),(80,0.25))
    def test_walrus_containing_scope(self):
        self.assertEqual(walrus_leakage(),([2,4,6],6))
    def test_empty_walrus_stale(self):
        self.assertEqual(empty_walrus(),([],"stale"))
    def test_valid_zero(self):
        self.assertEqual(update_if_present(5.0,0.0),0.0)
    def test_missing_sentinel(self):
        self.assertIs(metrics.get('missing',EMPTY),EMPTY)
    def test_stale_cache(self):
        self.assertEqual(invalid_cache(3),15)
    def test_correct_cache_key(self):
        self.assertEqual(valid_cache(3,7),21)
    def test_exception_cleanup(self):
        self.assertEqual(exception_binding_deletion(),"handler-binding-cleared")
    def test_pattern_guard(self):
        self.assertIsNone(guarded_match({'ticker':'A','price':-1}))
    def test_builtin_shadow(self):
        self.assertEqual(shadow_builtin(),'shadowed')
    def test_deterministic_lost_update(self):
        self.assertEqual(lost_update_demo(),1)
    def test_synchronized_update(self):
        self.assertEqual(synchronized_demo(),2)
    def test_equal_immutable_keys(self):
        self.assertEqual(result_cache[AssetKey('AAPL','NASDAQ')],100)
    def test_nan_not_equal_self(self):
        self.assertNotEqual(float('nan'),float('nan'))
    def test_inspector_findings(self):
        self.assertEqual(len(inspector.findings),3)

print('Language regression tests defined')

Language regression tests defined


### **Analytical Interpretation**

**1. Failure-first testing** — Each test documents a concrete Python behavior, including intentionally unsafe forms whose observed consequences are asserted.

**2. Reproducibility** — Tests reconstruct local mutable state where needed instead of assuming a previous notebook cell left an object untouched.

**3. Diagnostics** — The AST checks are deliberately limited to three source patterns; they are not a complete static-analysis engine.

**4. Finance relevance** — Demonstrating the failure mode is necessary before asserting that a correction truly removes cross-run contamination.

In [21]:
class ResearchExperimentIntegrityTests(unittest.TestCase):
    def setUp(self):
        self.raw = make_synthetic_prices()
        self.market = MarketSnapshot.from_prices(self.raw)
        self.spec = ExperimentSpec('AAPL_L2','AAPL',2,D('0.015'),D('5'))
        self.registry = ResearchRegistry()

    def test_snapshot_observation_count(self):
        self.assertEqual(len(self.market.observations),18)
    def test_session_count(self):
        self.assertEqual(self.market.session_count,9)
    def test_snapshot_fingerprint_length(self):
        self.assertEqual(len(self.market.fingerprint),64)
    def test_snapshot_deterministic(self):
        self.assertEqual(self.market,MarketSnapshot.from_prices(make_synthetic_prices()))
    def test_snapshot_isolation(self):
        self.raw['AAPL'][-1]=D('500')
        self.assertEqual(self.market.price_series('AAPL')[-1],D(113))
    def test_changed_source_fingerprint(self):
        self.raw['AAPL'][-1]=D('500')
        self.assertNotEqual(self.market.fingerprint,MarketSnapshot.from_prices(self.raw).fingerprint)
    def test_unknown_asset(self):
        with self.assertRaises(KeyError): self.market.price_series('ZZZ')
    def test_no_float_prices(self):
        self.raw['AAPL'][1]=103.0
        with self.assertRaises(ValueError): MarketSnapshot.from_prices(self.raw)
    def test_no_boolean_prices(self):
        self.raw['AAPL'][1]=True
        with self.assertRaises(ValueError): MarketSnapshot.from_prices(self.raw)
    def test_no_nan_prices(self):
        self.raw['AAPL'][1]=D('NaN')
        with self.assertRaises(ValueError): MarketSnapshot.from_prices(self.raw)
    def test_no_zero_prices(self):
        self.raw['AAPL'][1]=D(0)
        with self.assertRaises(ValueError): MarketSnapshot.from_prices(self.raw)
    def test_mismatched_history_length(self):
        self.raw['AAPL'].pop()
        with self.assertRaises(ValueError): MarketSnapshot.from_prices(self.raw)
    def test_invalid_ticker(self):
        self.raw['aapl']=self.raw.pop('AAPL')
        with self.assertRaises(ValueError): MarketSnapshot.from_prices(self.raw)
    def test_frozen_price_record(self):
        with self.assertRaises(FrozenInstanceError):
            self.market.observations[0].close_usd=D(1)
    def test_spec_invalid_lookback(self):
        with self.assertRaises(ValueError): ExperimentSpec('a','AAPL',True,D('0.01'),D(1))
    def test_spec_invalid_threshold(self):
        with self.assertRaises(ValueError): ExperimentSpec('a','AAPL',2,D(0),D(1))
    def test_spec_invalid_fee(self):
        with self.assertRaises(ValueError): ExperimentSpec('a','AAPL',2,D('0.01'),D(-1))
    def test_spec_invalid_id(self):
        with self.assertRaises(ValueError): ExperimentSpec('  ','AAPL',2,D('0.01'),D(1))
    def test_insufficient_lookback(self):
        spec=ExperimentSpec('x','AAPL',7,D('0.02'),D(1))
        with self.assertRaises(ValueError): evaluate_experiment(self.market,spec)
    def test_reference_trade_count(self):
        self.assertEqual(len(evaluate_experiment(self.market,self.spec).evaluations),3)
    def test_reference_gross(self):
        actual=evaluate_experiment(self.market,self.spec).gross_pnl_usd
        self.assertAlmostEqual(float(actual),485.719888,places=5)
    def test_reference_fees(self):
        self.assertEqual(evaluate_experiment(self.market,self.spec).total_cost_usd,D(15))
    def test_reference_net(self):
        actual=evaluate_experiment(self.market,self.spec).net_pnl_usd
        self.assertAlmostEqual(float(actual),470.719888,places=5)
    def test_pnl_identity(self):
        result=evaluate_experiment(self.market,self.spec)
        self.assertAlmostEqual(float(result.net_pnl_usd),float(result.gross_pnl_usd-result.total_cost_usd),places=9)
    def test_temporal_invariant(self):
        for trade in evaluate_experiment(self.market,self.spec).evaluations:
            self.assertTrue(trade.decision_session<trade.entry_session<trade.exit_session)
    def test_zero_trades_none_return(self):
        spec=ExperimentSpec('no','AAPL',2,D('0.99'),D(1))
        result=evaluate_experiment(self.market,spec)
        self.assertEqual(result.evaluations,())
        self.assertIsNone(result.mean_net_return_bps)
    def test_evaluation_deterministic(self):
        self.assertEqual(evaluate_experiment(self.market,self.spec),evaluate_experiment(self.market,self.spec))
    def test_registry_empty_batch(self):
        with self.assertRaises(ValueError): self.registry.submit_batch(self.market,())
    def test_registry_initial_snapshot(self):
        self.assertEqual(self.registry.snapshot(),RegistrySnapshot(0,()))
    def test_registry_successful_commit(self):
        result=self.registry.submit_batch(self.market,(self.spec,))
        self.assertEqual(result.version,1)
        self.assertEqual(len(result.results),1)
    def test_duplicate_within_batch(self):
        with self.assertRaises(ValueError): self.registry.submit_batch(self.market,(self.spec,self.spec))
        self.assertEqual(self.registry.snapshot().version,0)
    def test_duplicate_committed_id(self):
        self.registry.submit_batch(self.market,(self.spec,))
        before=self.registry.snapshot()
        with self.assertRaises(ValueError): self.registry.submit_batch(self.market,(self.spec,))
        self.assertEqual(self.registry.snapshot(),before)
    def test_failure_discards_first_staged_result(self):
        valid=ExperimentSpec('valid','AAPL',2,D('0.015'),D(5))
        invalid=ExperimentSpec('bad','AAPL',7,D('0.015'),D(5))
        with self.assertRaises(ValueError): self.registry.submit_batch(self.market,(valid,invalid))
        self.assertEqual(self.registry.snapshot(),RegistrySnapshot(0,()))
    def test_non_spec_rejection(self):
        with self.assertRaises(TypeError): self.registry.submit_batch(self.market,(self.spec, 'bad'))
        self.assertEqual(self.registry.snapshot().version,0)
    def test_cross_experiment_independence(self):
        other=ExperimentSpec('AAPL_L3','AAPL',3,D('0.025'),D(8))
        result=self.registry.submit_batch(self.market,(self.spec,other))
        lookup=dict(result.results)
        self.assertNotEqual(lookup['AAPL_L2'].net_pnl_usd,lookup['AAPL_L3'].net_pnl_usd)
        self.assertEqual(lookup['AAPL_L2'].spec.lookback,2)
        self.assertEqual(lookup['AAPL_L3'].spec.lookback,3)
    def test_registry_version_increment_once_per_batch(self):
        other=ExperimentSpec('other','AAPL',3,D('0.02'),D(2))
        self.registry.submit_batch(self.market,(self.spec,other))
        self.assertEqual(self.registry.snapshot().version,1)
    def test_prior_snapshot_unchanged(self):
        before=self.registry.snapshot()
        self.registry.submit_batch(self.market,(self.spec,))
        self.assertEqual(before,RegistrySnapshot(0,()))
    def test_result_is_frozen(self):
        result=evaluate_experiment(self.market,self.spec)
        with self.assertRaises(FrozenInstanceError): result.net_pnl_usd=D(0)
    def test_evaluation_cost_is_nonnegative(self):
        result=evaluate_experiment(self.market,self.spec)
        self.assertTrue(all(r.transaction_cost_usd>=0 for r in result.evaluations))
    def test_signed_direction(self):
        other=ExperimentSpec('msft','MSFT',2,D('0.01'),D(6))
        result=evaluate_experiment(self.market,other)
        self.assertTrue(any(r.direction=='SHORT' for r in result.evaluations))
    def test_no_mutation_of_market_on_evaluate(self):
        before=self.market
        evaluate_experiment(self.market,self.spec)
        self.assertEqual(self.market,before)
    def test_no_mutation_of_raw_on_evaluate(self):
        before={key:values.copy() for key,values in self.raw.items()}
        evaluate_experiment(self.market,self.spec)
        self.assertEqual(self.raw,before)

def execute_suite(*test_classes):
    suite=unittest.TestSuite(
        unittest.defaultTestLoader.loadTestsFromTestCase(cls)
        for cls in test_classes
    )
    result=unittest.TextTestRunner(verbosity=1).run(suite)
    if not result.wasSuccessful():
        raise AssertionError('One or more notebook tests failed')
    print(f'ALL {result.testsRun} AUTOMATED TESTS PASSED')
    return result.testsRun

VERIFIED_TEST_COUNT = execute_suite(
    AssignmentPitfallRegressionTests,
    ResearchExperimentIntegrityTests,
)

.

.

.

.

.

.

... (69 outputs left out)

.

.

----------------------------------------------------------------------
Ran 77 tests in 0.025s

OK


ALL 77 AUTOMATED TESTS PASSED


### **Analytical Interpretation**

**1. Language invariants** — The first suite validates the behaviors that create the pitfalls, as well as the selected remedies.

**2. Financial invariants** — The second suite independently checks reference gross P&L, transaction fees, net P&L, and temporal ordering.

**3. Data provenance** — Snapshot and fingerprint tests distinguish a preserved historical input from a revised raw source.

**4. Rollback and idempotency** — Duplicate IDs and later invalid specifications leave the committed registry unchanged.

**5. Test limitations** — The test suite does not establish statistical predictive ability, hardened security, or crash-safe multi-process transactions.

## 2.10.18 Final Analytical Interpretation

### **Analytical Interpretation**

**1. The central failure class is hidden state.** A function can execute successfully while observing a shared default, a captured mutable object, an outdated cache, or a stale name left over from earlier execution.

**2. Names are not independent storage.** Additional bindings do not copy objects; list multiplication and `dict.fromkeys` may multiply references rather than data structures.

**3. Assignment can fail after mutation.** Augmented assignment and multi-target assignment can change mutable objects before a later target rejects the update. Exception handling is not automatic rollback.

**4. Evaluation timing matters.** Function defaults capture definition-time objects; closures normally resolve free variables at call time; walrus expressions may bind into containing scopes; empty loops do not create new bindings.

**5. Read-only does not necessarily mean frozen.** A proxy over a mutable dictionary can reflect changes made elsewhere. Isolation requires a valid ownership and copying strategy.

**6. Static and dynamic verification are complementary.** AST inspection can flag syntactic risk patterns; only runtime tests can verify actual aliasing, effects, and financial outcomes for given inputs.

**7. Missingness must be explicit.** Zero P&L, zero exposure, `False`, and absent values have different semantics. Sentinels and explicit `None` checks prevent loss of valid data.

**8. Cache identity is a research contract.** Caches and experiment IDs must encode economically material dependencies, including input data versions and configuration parameters.

**9. Concurrency is a separate consistency problem.** Read–modify–write operations need an explicit synchronization policy. Interpreter-level implementation behavior is not an application transaction guarantee.

**10. Quant research requires lineage and staged publication.** In the capstone, immutable market snapshots isolate raw prices; frozen specifications separate parameters; staged batch commits prevent partial publication of results.

**11. Reference results are not evidence of alpha.** Synthetic signal-level P&L is an accounting and language demonstration; assessing actual predictive returns requires real market data, realistic execution, out-of-sample validation, and portfolio financing assumptions.

**12. Operational limitations remain.** The in-memory registry cannot guarantee durability across crashes, multi-process isolation, persistent idempotency, or distributed reconciliation. Additional infrastructure and financial controls would be necessary.

### Practical Code Review Checklist

- Are mutable defaults intentionally shared? If so, is the lifetime documented?
- Do closures capture each intended parameter value?
- Does each asset or scenario own its nested mutable containers?
- Are copied data genuinely isolated at required depths?
- Can an assignment raise after mutating an object?
- What happens to streams and targets after failed unpacking?
- Could `:=` or an empty loop expose a stale name?
- Is `0` a valid financial value rather than a missing value?
- Does the cache key include market-data version and full configuration?
- Are multi-field updates staged and committed under a coherent policy?
- Are snapshots independent of original mutable sources?
- Are tests run from a clean interpreter state?

**Final assessment:** Expert-level Python assignment literacy is the ability to predict **what changes, when it changes, who can observe it, and what remains changed after failure**. Those questions directly determine the reliability of quantitative research and stateful software.

## Advanced Practice Exercises — 12 Independent Challenges

### Exercise 1 — Definition-Time Capture and Research Contamination
Implement a research results collector with a deliberately mutable default. Reproduce cross-run contamination, then fix it with an explicit omission sentinel. Show that the corrected implementation still respects an intentionally supplied caller-owned list. Add tests that distinguish omission, `None`, and an empty provided list.

### Exercise 2 — Dataclass Deep-Immutability Audit
Construct a hierarchy of frozen dataclasses containing nested mappings, lists, and tuples. Identify all mutation routes still available through public references. Replace unsafe internals with validated immutable representations. Explain which assumptions cannot be guaranteed in hostile runtime code.

### Exercise 3 — Grid-Search Closure Capture
Write a parameter sweep that returns 30 callable signal functions using two thresholds and five lookbacks. Deliberately introduce late binding and prove the results collapse. Fix the problem using a factory, default capture, and `functools.partial`; verify equivalence of all corrected implementations.

### Exercise 4 — Object Graph Aliasing Inspector
Build an identity-based graph audit for nested dict/list/tuple/set structures. Detect repeated mutable references, including cycles, without infinite recursion. Report object paths and explain limitations involving custom objects, extensions, and weak references.

### Exercise 5 — Snapshot Isolation Under Revision
Construct a market-data snapshot system that survives later corrections to the original source. Compare shallow copy, deep copy, read-only proxy, and independent frozen records. Verify version changes and preserve original analytical results.

### Exercise 6 — Mutation-Before-Failure Matrix
Investigate augmented assignment for list, tuple-contained list, property getter/setter, custom `__iadd__`, and subscription targets. For each case, trace evaluation, in-place behavior, assignment-back, exceptions, and final state. Propose a safer staged alternative.

### Exercise 7 — Streaming Unpacking and Recovery
Implement a record parser that consumes a generator, intentionally produces unpacking mismatches, and reports lost records. Introduce bounded framing and retryable offsets. Demonstrate how exception handling alone fails to rewind the source.

### Exercise 8 — Assignment Expression and Stale-State Detection
Use `:=` in list comprehensions, generators, conditions, and loops. Design tests with zero iterations, falsy results, and exceptions. Distinguish newly produced values from stale bindings and document Python's applicable scope restrictions.

### Exercise 9 — Nullable Financial Metrics
Create a typed metrics API where zero, `False`, empty values, and missing records all have distinct meanings. Implement `None` and unique-sentinel handling without truthiness fallbacks. Verify edge cases for NaN, infinity, and signed zero.

### Exercise 10 — Cache and Data-Version Integrity
Build a multi-parameter memoization layer for quantitative signals. Show how excluding transaction costs or dataset versions from a cache key yields misleading repeated outcomes. Introduce complete immutable cache keys and input fingerprints; test cache invalidation across data revisions.

### Exercise 11 — Static Assignment-Pitfall Linter
Extend the AST inspector to analyze more function forms, starred assignments, annotated assignments, `AugAssign`, and suspicious mutable class attributes. Report source locations, false-positive examples, and limitations. Do not execute analyzed source code.

### Exercise 12 — Atomic Research Registry Under Concurrency
Extend the capstone registry with concurrent submissions, persistent experiment IDs, deterministic transaction replay, and durable storage. Compare thread safety, application-level batch atomicity, crash durability, and distributed consistency. Write failure-injection tests and define which guarantees are actually achieved.

### Submission Requirements

For every exercise, include: formal behavioral contract, deliberately failing example, corrected implementation, test suite, independent Analytical Interpretation, asymptotic complexity where relevant, and a short Quant Research risk assessment.

## Expert Challenge — Reproducible Multi-Strategy Experiment Infrastructure

### Objective

Design a reusable Python research platform in which **no experiment can silently alter another experiment's input data, parameters, intermediate state, or committed outputs**. The system must make assignment and ownership semantics explicit and testable.

### Architecture Deliverables

1. **Market Data Provenance:** immutable point-in-time observations, ingestion validation, source identifiers, content hashes, revision tracking, and explicit session-alignment policies.
2. **Experiment Specification:** frozen parameter objects encoding asset universe, lookbacks, thresholds, cost assumptions, numerical precision, and versioned code/model identifiers.
3. **Parameter Sweep Construction:** safe callback capture, immutable configuration, no mutable defaults, deterministic ordering, and traceable random seeds if stochastic algorithms are added.
4. **Feature Layer:** pure feature calculation, explicit point-in-time information sets, leakage checks, and independent per-asset intermediate buffers.
5. **Execution Evaluation:** separate signal time, next-session hypothetical entry, future outcomes, cost modeling, and limits on capital interpretation.
6. **State and Cache Isolation:** complete cache keys, invalidation rules, immutable results, explicit ownership policy, and contamination-detection tests.
7. **Batch Registry:** duplicate detection, staged evaluation, fail-fast validation, per-batch versioning, stable snapshots, synchronization, and retry semantics.
8. **Durable Storage:** documented atomic commits to a persistent store, crash-recovery procedure, replayable event history, and integrity verification.
9. **Quality Engineering:** mypy/pyright review, Ruff/PEP 8, unit tests, property-based tests, numerical reconciliation, controlled concurrency tests, and CI.
10. **Scientific Validation:** out-of-sample splits, multiple-testing awareness, trading-calendar rules, realistic execution assumptions, and comparison with suitable benchmarks.

### Suggested Project Layout

```text
advanced_assignment_integrity/
├── pyproject.toml
├── src/
│   └── research_integrity/
│       ├── data_models.py
│       ├── provenance.py
│       ├── parameter_binding.py
│       ├── features.py
│       ├── evaluation.py
│       ├── registry.py
│       ├── replay.py
│       └── diagnostics.py
├── tests/
│   ├── test_language_semantics.py
│   ├── test_aliasing.py
│   ├── test_feature_timing.py
│   ├── test_financial_invariants.py
│   ├── test_registry_rollback.py
│   └── test_concurrency.py
└── notebooks/
    └── 2.10_Advanced_Assignment_Pitfalls.ipynb
```

### Evaluation Rubric

| Dimension | Weight |
|---|---:|
| Python semantics and binding correctness | 25% |
| State isolation and reproducibility | 25% |
| Financial and temporal correctness | 20% |
| Failure-path and concurrency testing | 15% |
| Software architecture, typing, documentation | 15% |

### Completion Criteria

- No test depends on prior notebook execution state.
- All intended parameter values survive callback construction.
- Invalid input does not produce a committed experiment.
- Raw-data mutations do not change historical snapshot results.
- Failed multi-experiment batches preserve the committed registry.
- Every numerical metric has a documented unit and precision convention.
- Every claim of atomicity, durability, or thread safety is backed by a specific implementation mechanism and tests.

### References (Official Python Documentation)

- Python Language Reference — [Simple statements](https://docs.python.org/3/reference/simple_stmts.html)
- Python Language Reference — [Compound statements](https://docs.python.org/3/reference/compound_stmts.html)
- Python Language Reference — [Expressions](https://docs.python.org/3/reference/expressions.html)
- Python Library — [`dataclasses`](https://docs.python.org/3/library/dataclasses.html)
- Python Library — [`copy`](https://docs.python.org/3/library/copy.html)
- Python Library — [`ast`](https://docs.python.org/3/library/ast.html)
- Python Library — [`contextvars`](https://docs.python.org/3/library/contextvars.html)

**End of Section 2.10 — Advanced Assignment Pitfalls.**